# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [5]:
!pip install wget
import wget

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9685 sha256=8cb5d2060afa39bdab03b9778a8f7b4060a866262238413513d0c0c9af43c37c
  Stored in directory: /Users/chandlercampbell/Library/Caches/pip/wheels/01/46/3b/e29ffbe4ebe614ff224bad40fc6a5773a67a163251585a13a9
Successfully built wget


In [9]:
url = "https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip"
wget.download(url, "enron_spam_data.zip")

'enron_spam_data.zip'

In [10]:
import pandas as pd

In [14]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip", compression='zip')
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [15]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [ ]:
# YOUR CHANGES HERE

import re
import json
from collections import Counter

word_pattern = re.compile(r"[a-z0-9']+")
url_pattern = re.compile(r"(http[s]?://|www\.)", re.IGNORECASE)
caps_pattern = re.compile(r"\b[A-Z]{2,}\b")

def tokenize(text):
    return word_pattern.findall(text.lower())

def extract_features(subject, message):
    subject = "" if pd.isna(subject) else str(subject)
    message = "" if pd.isna(message) else str(message)

    subject_tokens = tokenize(subject)
    message_tokens = tokenize(message)

    feature_counts = Counter()

    # Count subject words
    for token in subject_tokens:
        if len(token) > 1:
            feature_counts[token] += 1
            feature_counts["subj_" + token] += 1

    # Count message words
    for token in message_tokens:
        if len(token) > 1:
            feature_counts[token] += 1

    # A few simple count features that often help spam detection
    combined_text = subject + " " + message

    num_exclam = combined_text.count("!")
    num_dollar = combined_text.count("$")
    num_urls = len(url_pattern.findall(combined_text))
    num_caps_words = len(caps_pattern.findall(combined_text))
    num_digits = sum(character.isdigit() for character in combined_text)

    if num_exclam > 0:
        feature_counts["num_exclam"] = num_exclam
    if num_dollar > 0:
        feature_counts["num_dollar"] = num_dollar
    if num_urls > 0:
        feature_counts["num_urls"] = num_urls
    if num_caps_words > 0:
        feature_counts["num_caps_words"] = num_caps_words
    if num_digits > 0:
        feature_counts["num_digits"] = num_digits

    return dict(feature_counts)

feature_rows = []

for row in enron_spam_data[["Message ID", "Subject", "Message"]].itertuples(index=False):
    message_id = int(row[0])
    subject = row[1]
    message = row[2]

    features = extract_features(subject, message)

    feature_rows.append({
        "Message ID": message_id,
        "features_json": json.dumps(features, sort_keys=True)
    })

all_features = pd.DataFrame(feature_rows)

train_features = all_features[all_features["Message ID"] % 30 != 0].copy()
test_features = all_features[all_features["Message ID"] % 30 == 0].copy()

train_features.to_csv("train-features.tsv", sep="\t", index=False)
test_features.to_csv("test-features.tsv", sep="\t", index=False)

print("Train shape:", train_features.shape)
print("Test shape:", test_features.shape)
print(train_features.head())
print(test_features.head())

Train shape: (32592, 2)
Test shape: (1124, 2)
   Message ID                                      features_json
1           1  {"00": 2, "000": 18, "09": 2, "10": 11, "108":...
2           2  {"calpine": 2, "daily": 2, "doc": 1, "gas": 2,...
3           3  {"01": 1, "06": 1, "08": 1, "09": 2, "10": 3, ...
4           4  {"01": 2, "02": 1, "09": 1, "10": 2, "12": 5, ...
5           5  {"10": 1, "11": 1, "19": 1, "3404": 1, "3405":...
     Message ID                                      features_json
0             0  {"christmas": 1, "farm": 1, "pictures": 1, "su...
30           30  {"04": 1, "07": 1, "11": 1, "12": 2, "17": 1, ...
60           60  {"7342": 1, "any": 1, "as": 1, "backup": 1, "b...
90           90  {"000": 1, "110": 1, "2000": 1, "22": 1, "33":...
120         120  {"01": 5, "05": 1, "09": 1, "11": 6, "12": 3, ...


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [17]:
# YOUR CHANGES HERE

alpha = 1

train_features = pd.read_csv("train-features.tsv", sep="\t")

train_labels = enron_spam_data[enron_spam_data["Message ID"] % 30 != 0][["Message ID", "Spam/Ham"]].copy()

train_merged = train_features.merge(train_labels, on="Message ID", how="inner")

ham_feature_counts = Counter()
spam_feature_counts = Counter()

total_ham_token_count = 0
total_spam_token_count = 0

for row in train_merged.itertuples(index=False):
    features_dict = json.loads(row.features_json)
    label = row._2

    if label == "ham":
        for feature_name, feature_count in features_dict.items():
            ham_feature_counts[feature_name] += int(feature_count)
            total_ham_token_count += int(feature_count)
    else:
        for feature_name, feature_count in features_dict.items():
            spam_feature_counts[feature_name] += int(feature_count)
            total_spam_token_count += int(feature_count)

all_features = sorted(set(ham_feature_counts.keys()) | set(spam_feature_counts.keys()))
vocabulary_size = len(all_features)

probability_rows = []

for feature_name in all_features:
    ham_count = ham_feature_counts.get(feature_name, 0)
    spam_count = spam_feature_counts.get(feature_name, 0)

    ham_probability = (ham_count + alpha) / (total_ham_token_count + alpha * vocabulary_size)
    spam_probability = (spam_count + alpha) / (total_spam_token_count + alpha * vocabulary_size)

    probability_rows.append({
        "feature": feature_name,
        "ham_probability": ham_probability,
        "spam_probability": spam_probability
    })

feature_probabilities = pd.DataFrame(probability_rows)

feature_probabilities.to_csv(
    "feature-probabilities.tsv",
    sep="\t",
    index=False,
    float_format="%.12f"
)

print("Number of features:", vocabulary_size)
print("Total ham token count:", total_ham_token_count)
print("Total spam token count:", total_spam_token_count)
print(feature_probabilities.head())

Number of features: 172414
Total ham token count: 4996855
Total spam token count: 3824604
    feature  ham_probability  spam_probability
0        00         0.001291      1.477101e-03
1       000         0.000890      1.331493e-03
2      0000         0.000006      1.235921e-04
3    000000         0.000001      3.802835e-05
4  00000000         0.000001      2.501865e-07


Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [19]:
# YOUR CHANGES HERE

import math

train_features = pd.read_csv("train-features.tsv", sep="\t")
feature_probabilities = pd.read_csv("feature-probabilities.tsv", sep="\t")

train_labels = enron_spam_data[enron_spam_data["Message ID"] % 30 != 0][["Message ID", "Spam/Ham"]].copy()

ham_prior = (train_labels["Spam/Ham"] == "ham").mean()
spam_prior = (train_labels["Spam/Ham"] == "spam").mean()

ham_probability_lookup = dict(
    zip(feature_probabilities["feature"], feature_probabilities["ham_probability"])
)
spam_probability_lookup = dict(
    zip(feature_probabilities["feature"], feature_probabilities["spam_probability"])
)

prediction_rows = []

for row in train_features.itertuples(index=False):
    message_id = int(row[0])
    features_dict = json.loads(row.features_json)

    log_ham_score = math.log(ham_prior)
    log_spam_score = math.log(spam_prior)

    for feature_name, feature_count in features_dict.items():
        feature_count = int(feature_count)

        ham_feature_probability = ham_probability_lookup[feature_name]
        spam_feature_probability = spam_probability_lookup[feature_name]

        log_ham_score += feature_count * math.log(ham_feature_probability)
        log_spam_score += feature_count * math.log(spam_feature_probability)

    max_log_score = max(log_ham_score, log_spam_score)

    ham_unnormalized = math.exp(log_ham_score - max_log_score)
    spam_unnormalized = math.exp(log_spam_score - max_log_score)

    probability_total = ham_unnormalized + spam_unnormalized

    ham_posterior = ham_unnormalized / probability_total
    spam_posterior = spam_unnormalized / probability_total

    prediction_rows.append({
        "Message ID": message_id,
        "ham": ham_posterior,
        "spam": spam_posterior
    })

train_predictions = pd.DataFrame(prediction_rows)
train_predictions.to_csv(
    "train-predictions.tsv",
    sep="\t",
    index=False,
    float_format="%.12f"
)

print(train_predictions.head())
print(train_predictions.shape)

   Message ID  ham           spam
0           1  1.0  3.270673e-216
1           2  1.0   3.834328e-18
2           3  1.0  1.997361e-163
3           4  1.0  3.343448e-160
4           5  1.0   8.321666e-44
(32592, 3)


Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.

Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [21]:
alpha = 1

test_features = pd.read_csv("test-features.tsv", sep="\t")
train_features = pd.read_csv("train-features.tsv", sep="\t")

train_labels = enron_spam_data[enron_spam_data["Message ID"] % 30 != 0][["Message ID", "Spam/Ham"]].copy()
train_merged = train_features.merge(train_labels, on="Message ID", how="inner")

ham_prior = (train_labels["Spam/Ham"] == "ham").mean()
spam_prior = (train_labels["Spam/Ham"] == "spam").mean()

ham_feature_counts = Counter()
spam_feature_counts = Counter()

total_ham_token_count = 0
total_spam_token_count = 0

for row in train_merged.itertuples(index=False):
    features_dict = json.loads(row.features_json)
    label = row[2]

    if label == "ham":
        for feature_name, feature_count in features_dict.items():
            feature_count = int(feature_count)
            ham_feature_counts[feature_name] += feature_count
            total_ham_token_count += feature_count
    else:
        for feature_name, feature_count in features_dict.items():
            feature_count = int(feature_count)
            spam_feature_counts[feature_name] += feature_count
            total_spam_token_count += feature_count

all_features = sorted(set(ham_feature_counts.keys()) | set(spam_feature_counts.keys()))
vocabulary_size = len(all_features)

ham_probability_lookup = {}
spam_probability_lookup = {}

for feature_name in all_features:
    ham_count = ham_feature_counts.get(feature_name, 0)
    spam_count = spam_feature_counts.get(feature_name, 0)

    ham_probability_lookup[feature_name] = (ham_count + alpha) / (
        total_ham_token_count + alpha * vocabulary_size
    )
    spam_probability_lookup[feature_name] = (spam_count + alpha) / (
        total_spam_token_count + alpha * vocabulary_size
    )

default_ham_probability = alpha / (total_ham_token_count + alpha * vocabulary_size)
default_spam_probability = alpha / (total_spam_token_count + alpha * vocabulary_size)

prediction_rows = []

for row in test_features.itertuples(index=False):
    message_id = int(row[0])
    features_dict = json.loads(row[1])

    log_ham_score = math.log(ham_prior)
    log_spam_score = math.log(spam_prior)

    for feature_name, feature_count in features_dict.items():
        feature_count = int(feature_count)

        ham_feature_probability = ham_probability_lookup.get(feature_name, default_ham_probability)
        spam_feature_probability = spam_probability_lookup.get(feature_name, default_spam_probability)

        log_ham_score += feature_count * math.log(ham_feature_probability)
        log_spam_score += feature_count * math.log(spam_feature_probability)

    max_log_score = max(log_ham_score, log_spam_score)

    ham_unnormalized = math.exp(log_ham_score - max_log_score)
    spam_unnormalized = math.exp(log_spam_score - max_log_score)

    probability_total = ham_unnormalized + spam_unnormalized

    ham_posterior = ham_unnormalized / probability_total
    spam_posterior = spam_unnormalized / probability_total

    prediction_rows.append({
        "Message ID": message_id,
        "ham": ham_posterior,
        "spam": spam_posterior
    })

test_predictions = pd.DataFrame(prediction_rows)
test_predictions.to_csv("test-predictions.tsv", sep="\t", index=False, float_format="%.12f")

print(test_predictions.head())
print(test_predictions.shape)

   Message ID       ham           spam
0           0  0.014419   9.855812e-01
1          30  1.000000   4.085448e-88
2          60  1.000000   1.080343e-13
3          90  1.000000   2.716040e-43
4         120  1.000000  2.606283e-206
(1124, 3)


Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [22]:
# YOUR CHANGES HERE
test_predictions = pd.read_csv("test-predictions.tsv", sep="\t")

test_labels = enron_spam_data[
    enron_spam_data["Message ID"] % 30 == 0
][["Message ID", "Spam/Ham"]].copy()

roc_data = test_predictions.merge(test_labels, on="Message ID", how="inner")

roc_rows = []

for threshold_number in range(1, 100):
    threshold = threshold_number / 100

    predicted_spam = roc_data["spam"] >= threshold
    actual_spam = roc_data["Spam/Ham"] == "spam"
    actual_ham = roc_data["Spam/Ham"] == "ham"

    true_positives = ((predicted_spam) & (actual_spam)).sum()
    false_positives = ((predicted_spam) & (actual_ham)).sum()
    false_negatives = ((~predicted_spam) & (actual_spam)).sum()
    true_negatives = ((~predicted_spam) & (actual_ham)).sum()

    if (true_positives + false_negatives) > 0:
        true_positive_rate = true_positives / (true_positives + false_negatives)
    else:
        true_positive_rate = 0.0

    if (false_positives + true_negatives) > 0:
        false_positive_rate = false_positives / (false_positives + true_negatives)
    else:
        false_positive_rate = 0.0

    roc_rows.append({
        "threshold": threshold,
        "false_positive_rate": false_positive_rate,
        "true_positive_rate": true_positive_rate
    })

roc_table = pd.DataFrame(roc_rows)

roc_table.to_csv("roc.tsv", sep="\t", index=False, float_format="%.12f")

print(roc_table.head())
print(roc_table.tail())

   threshold  false_positive_rate  true_positive_rate
0       0.01             0.030797            0.998252
1       0.02             0.030797            0.998252
2       0.03             0.028986            0.998252
3       0.04             0.028986            0.998252
4       0.05             0.028986            0.998252
    threshold  false_positive_rate  true_positive_rate
94       0.95             0.018116            0.982517
95       0.96             0.018116            0.982517
96       0.97             0.018116            0.980769
97       0.98             0.018116            0.980769
98       0.99             0.014493            0.980769


Submit "roc.tsv" in Gradescope.

## Part 7: Signup for Gemini API Key

Create a free Gemini API key at https://aistudio.google.com/app/api-keys.
You will need to do this with a personal Google account - it will not work with your BU Google account.
This will not incur any charges unless you configure billing information for the key.

You will be asked to start a Gemini free trial for week 11.
This will not incur any charges unless you exceed expected usage by an order of magnitude.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.